# Liora Engine · ALTERNATIVA Wan 2.1 VACE 1.3B en Kaggle (0 €)

**Solo si Wan 2.2 5B falla o resulta inviable en las T4.** Mismo procedimiento: *GPU T4 x2* + *Internet On* →
**Save Version → Save & Run All (Commit)**.

Wan 2.1 VACE 1.3B (familia Wan, licencia leída de la ficha del modelo al descargar): 1,3 B parámetros en fp16 en **una sola T4**,
codificador UMT5 en CPU. Prueba mínima: **3,5 s (57 fotogramas) a 480×832 y 16 fps, 30 pasos**, mujer ficticia generada con texto;
después, imagen→vídeo desde un fotograma sintético. Admite imágenes de referencia de identidad (para Liora, solo en un dataset privado).
Menor resolución y calidad que Wan 2.2 5B. Nada se simula: los errores quedan en `evidence.json` y `run.log`.

In [ ]:
# 0) Detenerse ANTES de instalar o descargar nada si Kaggle no ha concedido GPU
import os, shutil, subprocess, pathlib
assert shutil.which('nvidia-smi'), 'No hay GPU NVIDIA asignada: Session options → Accelerator → GPU T4 x2, y reinicia.'
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
assert gpu.returncode == 0 and gpu.stdout.strip(), 'GPU NVIDIA no utilizable en esta sesión.'
import torch
assert torch.cuda.is_available(), 'PyTorch no detecta CUDA: no se descargan los pesos.'

In [ ]:
import os, pathlib
WORK = pathlib.Path('/kaggle/working' if os.path.isdir('/kaggle/working') else 'liora_work').resolve()
WORK.mkdir(exist_ok=True)
# Pesos fuera de /kaggle/working (límite de 20 GB de salida); /root tiene más espacio en Kaggle
os.environ.setdefault('HF_HOME', '/root/.cache/huggingface')
!nvidia-smi
!free -g; df -h /root "$WORK" 

In [ ]:
# Código del motor (siempre la última versión de main) y versiones exactas validadas con el motor
!rm -rf edge-funding && git clone -q --depth 1 --filter=blob:none --sparse https://github.com/alvarogavilan/edge-funding.git
!cd edge-funding && git sparse-checkout set liora/engine && git log -1 --format='commit %H %cd'
os.chdir('edge-funding/liora/engine')
# torch de Kaggle se conserva; diffusers 0.41 es la primera versión validada con componentes repartidos entre dispositivos
!pip install -q "diffusers==0.41.0" "transformers==5.20.1" "accelerate==1.15.0" "huggingface_hub==1.33.0" "tokenizers==0.23.3" "safetensors==0.8.0" ftfy sentencepiece imageio-ffmpeg
!python -c "import torch, diffusers, transformers; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.device_count(), '| diffusers', diffusers.__version__, '| transformers', transformers.__version__)" 

In [ ]:
# 1) Capacidad real de la GPU gratuita
!python -m liora_engine probe --bench --out "$WORK/capacity.json" | head -40

In [ ]:
# 2) Pesos de Wan 2.1 VACE 1.3B, con tiempo, tamaño y licencia leída de la ficha oficial del modelo
import time, subprocess
from huggingface_hub import snapshot_download, model_info
t0 = time.time()
MODEL = snapshot_download('Wan-AI/Wan2.1-VACE-1.3B-diffusers')
size = int(subprocess.check_output(['du', '-sbL', MODEL]).split()[0])
info = model_info('Wan-AI/Wan2.1-VACE-1.3B-diffusers')
print(MODEL, round(size / 1e9, 1), 'GB en', round(time.time() - t0), 's | licencia de la ficha:', getattr(info.card_data, 'license', None), '| commit', info.sha)
os.environ['LIORA_WAN21_VACE'] = MODEL

In [ ]:
# 3) PRUEBA (texto→vídeo, mujer ficticia): 3,5 s = 57 fotogramas a 480×832, 16 fps, 30 pasos
#    Escribe liora-smoke.mp4 + evidence.json + run.log en WORK
!python -m liora_engine smoke --work "$WORK" --backend wan21-vace-1.3b --width 480 --height 832 --fps 16 --seconds 3.5 --steps 30

In [ ]:
# 3b) Solo si 3) marcó «fotogramas INVÁLIDOS» (desbordamiento fp16): repetir en bfloat16 (emulado en T4, más lento)
import json
ev = json.load(open(WORK / 'evidence.json'))
print(ev['status'], ev.get('error'), ev.get('frame_health'))
if ev['status'] == 'invalid_output':
    os.environ['LIORA_DTYPE'] = 'bfloat16'
    !python -m liora_engine smoke --work "$WORK" --backend wan21-vace-1.3b --width 480 --height 832 --fps 16 --seconds 3.5 --steps 30

In [ ]:
# 4) Evidencia y reproducción
import json
from IPython.display import Video
ev = json.load(open(WORK / 'evidence.json'))
print(json.dumps({k: ev.get(k) for k in ('status', 'generated', 'generation_s', 'total_s', 'model_revision', 'params', 'mp4', 'frame_health')}, indent=2, ensure_ascii=False))
print(json.dumps(ev['hardware']['gpu'], indent=2))
Video(str(WORK / 'liora-smoke.mp4'), embed=True, width=360) if ev.get('generated') else print('Sin MP4 válido: ver error/preflight en evidence.json')

In [ ]:
# 5) Verificación independiente con ffprobe (no depende del motor)
import json, hashlib, subprocess, shutil
clip = WORK / 'liora-smoke.mp4'
assert clip.exists() and clip.stat().st_size > 10000, 'No existe un MP4 válido: la prueba P0 NO ha concluido.'
if not shutil.which('ffprobe'):
    !apt-get -qq install -y ffmpeg > /dev/null 2>&1
info = json.loads(subprocess.check_output(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-count_frames',
    '-show_entries', 'stream=codec_name,profile,pix_fmt,width,height,r_frame_rate,nb_read_frames,duration', '-of', 'json', str(clip)]))
print(json.dumps({'file': clip.name, 'bytes': clip.stat().st_size, 'sha256': hashlib.sha256(clip.read_bytes()).hexdigest(), **info['streams'][0]}, indent=2))

In [ ]:
# 6) Imagen→vídeo desde un fotograma SINTÉTICO del clip anterior (no es la foto de Liora): valida el modo que usarán las escenas
import json, subprocess
ev = json.load(open(WORK / 'evidence.json'))
if ev.get('generated'):
    from liora_engine import media
    media.first_frame(WORK / 'liora-smoke.mp4', WORK / 'sintetica.png')
    !python -m liora_engine smoke --work "$WORK/i2v" --backend wan21-vace-1.3b --width 480 --height 832 --fps 16 --seconds 3.5 --steps 30 --image "$WORK/sintetica.png"
else:
    print('Primero debe existir un MP4 válido en el paso 3.')